# 07 — Explainability Layer

Global SHAP feature importance, local explanations for individual loans, false positive / false negative case studies, calibration curves, and model confidence.

In [ ]:
import sys, warnings, json
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

# Load global importance
gi = json.load(open('../reports/explainability/global_importance.json'))
targets = list(gi.keys())
print('Targets with SHAP importance:', targets)


## Global Feature Importance (Top 15) — Default Model

In [ ]:
target = 'next_12m_default_flag'
if target in gi and 'top_20' in gi[target]:
    imp = pd.Series(gi[target]['top_20']).sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(8, 5))
    imp.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Global SHAP Feature Importance — {target}')
    ax.set_xlabel('Mean |SHAP value|')
    plt.tight_layout()
    plt.savefig('../reports/shap_global_importance_default.png', dpi=150)
    plt.show()
else:
    print('Run the full pipeline to generate SHAP values.')


## SHAP Summary Plots (pre-generated)

In [ ]:
from IPython.display import Image, display
import os
plots_dir = Path('../reports/explainability/plots')
for f in sorted(plots_dir.glob('shap_summary_*.png'))[:4]:
    print(f'\n--- {f.stem} ---')
    display(Image(str(f)))


## False Positive / False Negative Analysis

In [ ]:
ea = json.load(open('../reports/explainability/error_analysis.json'))
for tgt, analysis in list(ea.items())[:2]:
    fps = analysis.get('false_positives', [])
    fns = analysis.get('false_negatives', [])
    print(f'\n{tgt}:')
    print(f'  False positives: {len(fps)}  |  False negatives: {len(fns)}')
    if fps:
        fp = fps[0]
        print(f'  Example FP — base_value={fp.get("base_value","?"):.4f}, prediction={fp.get("prediction","?"):.4f}')
        top5 = fp.get('contributions', [])[:3]
        for c in top5:
            print(f'    {c["feature"]}: {c["shap_value"]:+.4f}')


## Calibration Check

In [ ]:
# Load a model and check calibration on validation set
train_df = pd.read_csv('../data/synthetic/loan_monthly_performance_train.csv')
for col in ['reporting_month','origination_month']:
    if col in train_df.columns:
        train_df[col] = pd.to_datetime(train_df[col]).dt.to_period('M')
from src.features.engineer import LeakageSafeFeatureEngineer
from src.evaluation.time_split import get_split_masks
fe = LeakageSafeFeatureEngineer()
fe.load_artifacts('../models/feature_engineering')
X_full = fe.transform(train_df)
_, val_mask, _ = get_split_masks(train_df)
X_val = X_full[val_mask]
y_val = train_df['next_12m_default_flag'][val_mask]
model = joblib.load('../models/classification/next_12m_default_flag_improved.pkl')
y_proba = model.predict_proba(X_val)[:, 1]

from src.evaluation.metrics import calibration_summary
cal_df = calibration_summary(y_val.values, y_proba)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(cal_df['mean_predicted_prob'], cal_df['fraction_of_positives'], 'o-', label='Model')
ax.plot([0,1],[0,1],'--', color='gray', label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration — 12-Month Default Model')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/calibration_curve.png', dpi=150)
plt.show()
